## Implementazione di un Bot Discord per la Classificazione CIFAR-10
In questa sezione integriamo il modello addestrato con un Bot Discord. 
Il bot riceverà un'immagine, la processerà e restituirà la classe predetta.

In [2]:
# Installazione delle librerie necessarie
#!pip install discord.py tensorflow pillow nest-asyncio

import nest_asyncio
import io
import numpy as np
import tensorflow as tf
from PIL import Image
import discord
from discord.ext import commands

# Patch necessaria per far girare il loop di Discord dentro Jupyter
nest_asyncio.apply()  #Ho incluso nest_asyncio perché i file Jupyter hanno spesso conflitti con i loop di Discord.

##### 1. Caricamento del Modello e Definizione Classi
Carichiamo il file `.h5` generato durante la fase di training e definiamo le 10 etichette del dataset CIFAR-10.

In [3]:
# Sostituisci 'modello_cifar10.h5' con il nome del tuo file
MODEL_PATH = 'modello_cifar10.h5'
model = tf.keras.models.load_model(MODEL_PATH)

class_names = ['aereo', 'automobile', 'uccello', 'gatto', 'cervo', 
               'cane', 'rana', 'cavallo', 'nave', 'camion']

print("Modello caricato correttamente!")

OSError: Unable to synchronously open file (file signature not found)

# Preprocessing

## 2. Funzione di Pre-processing
Le immagini inviate su Discord possono avere qualsiasi dimensione. La funzione `prepare_image` si occupa di:
1. Convertire l'immagine in RGB.
2. Ridimensionarla a 32x32 pixel.
3. Normalizzare i valori dei pixel tra 0 e 1.

In [ ]:
def prepare_image(image_bytes):
    # Apre l'immagine dai byte ricevuti
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    
    # Ridimensiona per il modello CIFAR-10
    img = img.resize((32, 32))
    
    # Conversione in array e normalizzazione
    img_array = np.array(img) / 255.0
    
    # Aggiunta della dimensione batch (1, 32, 32, 3)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

# Configurazione e Comandi Bot

### 3. Configurazione del Bot Discord
Definiamo il prefisso del comando (`!`) e la logica per gestire l'allegato. 
Il bot scaricherà l'immagine, userà il modello per la predizione e risponderà all'utente.

In [ ]:
# Configurazione permessi (Intents)
intents = discord.Intents.default()
intents.message_content = True 
bot = commands.Bot(command_prefix="!", intents=intents)

#Intents: Sono i "sensori" del bot. Per motivi di privacy, 
# Discord non permette ai bot di leggere i messaggi degli utenti a meno che
# non venga richiesto esplicitamente. Impostando message_content = True,
# stai dicendo a Discord che il tuo bot ha il permesso di leggere il testo dei messaggi
# per vedere se contengono comandi.

#command_prefix="!": Questo definisce il carattere che "chiama" il bot.
#In questo caso, il bot ignorerà i messaggi normali e si attiverà solo se 
#un messaggio inizia con !.

@bot.event
async def on_ready():
    print(f'✅ Bot online come: {bot.user}')

@bot.command()
async def classifica(ctx):
    # Verifica se l'utente ha allegato un file
    if not ctx.message.attachments:
        await ctx.send("⚠️ Per favore, allega un'immagine al comando.")
        return
#Il codice controlla se nel messaggio c'è almeno un file allegato (attachments).
# Se non c'è, avvisa l'utente e si ferma (return).
    attachment = ctx.message.attachments[0]
    
    if any(attachment.filename.lower().endswith(ext) for ext in ['png', 'jpg', 'jpeg']):
        await ctx.send("🔎 Sto analizzando l'immagine...") 
#Qui il bot verifica che il file sia effettivamente una foto (estensione .png o .jpg,jpeg).

        
        # Download e Predizione
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        
        preds = model.predict(processed_img)
#attachment.read(): Scarica l'immagine da Discord e la trasforma in byte (dati grezzi).

#prepare_image: Passa questi dati alla funzione che abbiamo scritto nella cella precedente
#per ridimensionarli a 32x32 pixel e normalizzarli.

#model.predict: Passa l'immagine elaborata alla tua rete neurale.
        index = np.argmax(preds)
        accuracy = np.max(tf.nn.softmax(preds[0])) * 100
        label = class_names[index]
        await ctx.send(f"🤖 Risultato: **{label}** (Confidenza: {accuracy:.2f}%)")
    else:
        await ctx.send("❌ Formato non supportato. Invia una foto JPG o PNG.")

#np.argmax(preds): Il modello restituisce una lista di 10 probabilità (una per ogni classe). 

#argmax estrae l'indice del valore più alto (es. se la probabilità più alta è in posizione 3,
# il risultato sarà 'gatto').

#tf.nn.softmax: Trasforma i valori grezzi del modello in percentuali leggibili (0-100%).

#ctx.send: Il bot invia il messaggio finale nella chat di Discord con il nome della classe
#e quanto è sicuro della sua scelta.

### Stato Personalizzato
bot mostra cosa sta facendo sotto il suo nome nella lista utenti.

In [ ]:
@bot.event
async def on_ready():
    # Imposta lo stato: "In ascolto di !classifica"
    activity = discord.Activity(type=discord.ActivityType.listening, name="!classifica")
    await bot.change_presence(status=discord.Status.online, activity=activity)
    print(f'✅ Bot pronto: {bot.user}')

### Gestione Errori

In [ ]:
# 1. Configurazione Stato Personalizzato
@bot.event
async def on_ready():
    # Imposta lo stato: "In ascolto di !info"
    activity = discord.Activity(type=discord.ActivityType.listening, name="!info")
    await bot.change_presence(status=discord.Status.online, activity=activity)
    print(f'✅ Bot online come: {bot.user}')

# 2. Gestione Errori Globale
@bot.event
async def on_command_error(ctx, error):
    if isinstance(error, commands.CommandNotFound):
        await ctx.send("❓ Comando non riconosciuto. Scrivi `!info` per vedere cosa posso fare.")
    elif isinstance(error, commands.MissingRequiredArgument):
        await ctx.send("⚠️ Mancano degli argomenti nel comando.")
    else:
        print(f"Errore non gestito: {error}")

### Comando Classifica con Barra

In [ ]:
@bot.command()
async def classifica(ctx):
    # Controllo allegati
    if not ctx.message.attachments:
        await ctx.send("⚠️ Per favore, carica un'immagine insieme al comando `!classifica`.")
        return

    attachment = ctx.message.attachments[0]
    
    # Filtro estensioni
    if not any(attachment.filename.lower().endswith(ext) for ext in ['png', 'jpg', 'jpeg']):
        await ctx.send("❌ Formato file non supportato. Usa solo JPG o PNG.")
        return

    # Inizio caricamento grafico
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] 0% - Ricezione immagine...")
    
    try:
        # Download (40%)
        image_bytes = await attachment.read()
        await loading_msg.edit(content="⌛ [████░░░░░░] 40% - Elaborazione pixel...")
        
        # Preprocessing (70%)
        processed_img = prepare_image(image_bytes)
        await loading_msg.edit(content="⌛ [███████░░░] 70% - Interrogazione Rete Neurale...")
        
        # Predizione (100%)
        preds = model.predict(processed_img)
        await loading_msg.edit(content="⌛ [██████████] 100% - Analisi completata!")
        
        # Calcolo risultati
        index = np.argmax(preds)
        accuracy = np.max(tf.nn.softmax(preds[0])) * 100
        label = class_names[index]

        # Creazione dell'Embed finale
        embed = discord.Embed(
            title="🔍 Risultato del Riconoscimento",
            description=f"Analisi completata per l'utente {ctx.author.mention}",
            color=0x3498db # Colore blu elegante
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Predetto", value=f"**{label.upper()}**", inline=True)
        embed.add_field(name="Grado di Sicurezza", value=f"{accuracy:.2f}%", inline=True)
        embed.set_footer(text="AI Model: CIFAR-10 CNN | UniBari Project")

        # Rimuove la barra e invia il risultato
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Si è verificato un errore durante l'analisi: {e}")

### Comando Informativo
Aggiungiamo un comando per spiegare agli utenti quali sono le 10 categorie che il modello può riconoscere (dataset CIFAR-10) e come utilizzare il bot.

In [ ]:
@bot.command()
async def info(ctx):
    # Creiamo un messaggio formattato in modo elegante (Embed)
    descrizione = (
        "Ciao! Sono il Bot del progetto di Classificazione Immagini.\n"
        "Utilizzo una rete neurale (CNN) addestrata sul dataset **CIFAR-10**.\n\n"
        "**Cosa posso fare?**\n"
        "Se mi invii una foto e scrivi `!classifica`, proverò a capire cosa rappresenta.\n\n"
        "**Le mie 10 categorie sono:**\n"
        f"✈️ {class_names[0]}, 🚗 {class_names[1]}, 🐦 {class_names[2]}, 🐱 {class_names[3]}, 🦌 {class_names[4]},\n"
        f"🐶 {class_names[5]}, 🐸 {class_names[6]}, 🐴 {class_names[7]}, 🚢 {class_names[8]}, 🚛 {class_names[9]}.\n\n"
        "**Istruzioni:**\n"
        "Carica una foto e scrivi `!classifica` nel campo del commento!"
    )
    
    await ctx.send(descrizione)

# Esecuzione
Inserisci il tuo Token e avvia la cella. Il bot rimarrà attivo finché la cella è in esecuzione.

In [ ]:
TOKEN = 'MTQ4MDU1NzkxMjc5NDMzNzMzMA.GwAsuz.NLZ-9DomHKx2Rshu5bx1zoYdHeHDfhygw-zsHM'
bot.run(TOKEN)

[2026-03-09 12:40:57] [INFO    ] discord.client: logging in using static token
[2026-03-09 12:40:57] [INFO    ] discord.client: logging in using static token


LoginFailure: Improper token has been passed.

### Come ottenere il Token (Passaggi Fondamentali)
Prima di eseguire il codice, devi registrare il bot:

- Vai su Discord Developer Portal.

- Clicca su "New Application" e dai un nome.

- Vai a sinistra su "Bot".

- Clicca su "Reset Token" per visualizzare e copiare il tuo Token.

IMPORTANTE: Scorri verso il basso nella pagina Bot e attiva "Message Content Intent". Senza questo, il bot non leggerà il comando !classifica.

### Caricamento nella cartella del progetto
Una volta che hai verificato che tutto funziona localmente:

Puoi caricare il file .ipynb nella cartella del progetto su GitHub o Drive.

Ricordati di non caricare mai il Token in chiaro su GitHub pubblico (usa un file .env o cancellalo prima di caricare).